# Text-to-Mask Sanity Check\n\nGoal: run the first working `image + text prompt -> box -> mask` pipeline using Grounding DINO and SAM.\n

## 1. Check GPU\n

In [ ]:
import torch\n\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))\n\nDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\nDEVICE\n

## 2. Install dependencies\n\nThis cell may take several minutes.\n

In [ ]:
!pip install -q opencv-python matplotlib pillow numpy supervision\n!pip install -q git+https://github.com/facebookresearch/segment-anything.git\n\n!rm -rf GroundingDINO\n!git clone -q https://github.com/IDEA-Research/GroundingDINO.git\n%cd GroundingDINO\n!pip install -q -e .\n%cd ..\n

## 3. Download model weights\n

In [ ]:
!mkdir -p weights\n\n# Grounding DINO Swin-T checkpoint\n!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0/groundingdino_swint_ogc.pth -O weights/groundingdino_swint_ogc.pth\n\n# SAM ViT-B checkpoint\n!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O weights/sam_vit_b_01ec64.pth\n\n!ls -lh weights\n

## 4. Download demo image\n\nLater we will replace this with our own test images. For the first sanity check, one public image is enough.\n

In [ ]:
!mkdir -p data/demo_images outputs/predictions\n!wget -q https://raw.githubusercontent.com/IDEA-Research/Grounded-Segment-Anything/main/assets/demo1.jpg -O data/demo_images/demo1.jpg\n\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\nIMAGE_PATH = 'data/demo_images/demo1.jpg'\nimage_pil = Image.open(IMAGE_PATH).convert('RGB')\n\nplt.figure(figsize=(8, 8))\nplt.imshow(image_pil)\nplt.axis('off')\n

## 5. Load Grounding DINO\n

In [ ]:
import sys\nsys.path.append('/content/GroundingDINO')\n\nfrom groundingdino.util.inference import load_model, load_image, predict, annotate\n\nGROUNDING_DINO_CONFIG = '/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'\nGROUNDING_DINO_WEIGHTS = 'weights/groundingdino_swint_ogc.pth'\n\ngrounding_model = load_model(GROUNDING_DINO_CONFIG, GROUNDING_DINO_WEIGHTS)\ngrounding_model = grounding_model.to(DEVICE)\nprint('Grounding DINO loaded')\n

## 6. Run text-guided detection\n

In [ ]:
import cv2\nimport numpy as np\n\nTEXT_PROMPT = 'dog . person . car .'\nBOX_THRESHOLD = 0.30\nTEXT_THRESHOLD = 0.25\n\nimage_source, image_tensor = load_image(IMAGE_PATH)\n\nboxes, logits, phrases = predict(\n    model=grounding_model,\n    image=image_tensor,\n    caption=TEXT_PROMPT,\n    box_threshold=BOX_THRESHOLD,\n    text_threshold=TEXT_THRESHOLD,\n    device=DEVICE,\n)\n\nprint('boxes:', boxes.shape)\nprint('logits:', logits)\nprint('phrases:', phrases)\n\nannotated_frame = annotate(\n    image_source=image_source,\n    boxes=boxes,\n    logits=logits,\n    phrases=phrases,\n)\n\nannotated_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)\n\nplt.figure(figsize=(10, 10))\nplt.imshow(annotated_rgb)\nplt.axis('off')\nplt.title('Grounding DINO detections')\n

## 7. Load SAM\n

In [ ]:
from segment_anything import sam_model_registry, SamPredictor\n\nSAM_CHECKPOINT = 'weights/sam_vit_b_01ec64.pth'\nSAM_MODEL_TYPE = 'vit_b'\n\nsam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT)\nsam.to(device=DEVICE)\n\nsam_predictor = SamPredictor(sam)\nsam_predictor.set_image(image_source)\n\nprint('SAM loaded')\n

## 8. Convert Grounding DINO boxes and run SAM\n

In [ ]:
from groundingdino.util import box_ops\n\nH, W, _ = image_source.shape\n\n# Grounding DINO returns normalized cxcywh boxes. Convert to pixel xyxy boxes.\nboxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.Tensor([W, H, W, H])\nboxes_xyxy = boxes_xyxy.to(DEVICE)\n\ntransformed_boxes = sam_predictor.transform.apply_boxes_torch(\n    boxes_xyxy,\n    image_source.shape[:2],\n).to(DEVICE)\n\nmasks, mask_scores, _ = sam_predictor.predict_torch(\n    point_coords=None,\n    point_labels=None,\n    boxes=transformed_boxes,\n    multimask_output=False,\n)\n\nprint('masks:', masks.shape)\nprint('mask_scores:', mask_scores)\n

## 9. Visualize final text-to-mask result\n

In [ ]:
import matplotlib.patches as patches\n\ndef show_mask(mask, ax, alpha=0.45):\n    mask = mask.cpu().numpy()\n    if mask.ndim == 3:\n        mask = mask[0]\n    color = np.array([30 / 255, 144 / 255, 255 / 255, alpha])\n    h, w = mask.shape[-2:]\n    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)\n    ax.imshow(mask_image)\n\ndef show_box(box, ax, label=None):\n    x0, y0, x1, y1 = box.cpu().numpy()\n    rect = patches.Rectangle(\n        (x0, y0),\n        x1 - x0,\n        y1 - y0,\n        edgecolor='lime',\n        facecolor='none',\n        linewidth=2,\n    )\n    ax.add_patch(rect)\n    if label is not None:\n        ax.text(x0, y0 - 5, label, color='white', fontsize=10,\n                bbox=dict(facecolor='green', alpha=0.7))\n\nfig, ax = plt.subplots(figsize=(12, 12))\nax.imshow(image_source)\n\nfor i, box in enumerate(boxes_xyxy):\n    label = phrases[i] if i < len(phrases) else None\n    show_box(box, ax, label=label)\n    show_mask(masks[i], ax)\n\nax.axis('off')\nax.set_title(f'Text-to-mask result: {TEXT_PROMPT}')\nplt.savefig('outputs/predictions/sanity_text_to_mask.png', bbox_inches='tight', dpi=150)\nplt.show()\n\nprint('Saved to outputs/predictions/sanity_text_to_mask.png')\n